# L12d Lab: A Linear DQN for Adaptive Drug Dosing
In this lab, we learn a chemotherapy dosing policy for the patient of the [dosing lab](../L12b/CHEME-5800-L12b-Lab-AdaptiveDosing-QLearning-Fall-2026.ipynb) with the deep Q-network (DQN) method. In the dosing lab, value iteration and Q-learning worked on a grid of 41 levels per variable.

Here, the tumor burden and the toxicity are continuous numbers between 0 and 1. A linear Q-network computes the value of each dose from features of the patient state.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Follow grid policies on the continuous patient:__ A grid policy chooses each dose from the patient state rounded to the nearest grid point. We'll compute grid policies on the 41-level grid of the dosing lab and on finer grids, and follow them on a patient whose state is not rounded.
> * __Build a linear Q-network:__ A linear Q-network computes the value of each dose as a weighted sum of radial basis functions of the tumor burden and the toxicity. We'll build one and evaluate it at any patient state.
> * __Train the network with DQN:__ The DQN algorithm stores the agent's transitions in a replay buffer and fits the network to targets computed by the target network, a delayed copy of the network. We'll train the network in a few seconds and compare its doses with the grid policies.

Let's get started!
___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [ ]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

# Choose one theme; after changing it, rerun this cell and the plotting cells -
theme(:default)   # light background
# theme(:dark)   # dark background

In addition to standard Julia libraries, we'll use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

The lab's code is in the [src](src) folder, and it is complete. Each task ends with one question for the class, under __Things to think about__. Some questions ask us to change a value in a code cell and rerun it.
___

## Task 1: Build the Continuous Patient Model
In this task, we build the patient model and treat the patient for one cycle.

The patient state is the pair $(B,Z)$, where $B$ is the tumor burden and $Z$ is the accumulated toxicity, both scaled to lie between 0 and 1. Each treatment cycle, we choose a dose $u$ from $\{0, 1/3, 2/3, 1\}$. As in the dosing lab, the next state is given by:

$$
\begin{align*}
B^{\prime} & = B + g\,B\,(1-B) - k\,u\,B, \\
Z^{\prime} & = Z + b\,u - c\,Z,
\end{align*}
$$

where $g$ is the tumor growth rate, $k$ is the fraction of the tumor killed by a full dose, $b$ is the toxicity added by a full dose, and $c$ is the fraction of the toxicity cleared each cycle. Each new value is clipped to lie between 0 and 1.

The rewards are the same as in the dosing lab:
* A cure, $B^{\prime}\leq 0.10$ with the patient alive, earns 60 and ends the treatment.
* A toxic death, $Z^{\prime}\geq 0.90$, costs 60 and ends the treatment.
* Any other cycle costs $w_{B}\,B^{\prime}+w_{Z}\,Z^{\prime}+w_{u}\,u$, where $w_{B}$, $w_{Z}$, and $w_{u}$ are the penalty weights of the tumor burden, the toxicity, and the dose. We use $w_{B}=w_{Z}=1$ and $w_{u}=0.1$.

The one change from the dosing lab is that $B^{\prime}$ and $Z^{\prime}$ are not rounded to a grid level.

> __Why does the rounding matter?__ Suppose the patient at $(B,Z)=(0.90,0)$ gets the dose $u=2/3$. The equations give the next state $(B^{\prime},Z^{\prime})=(0.7035,0.3333)$.
>
> * __Dosing lab (rounded):__ The world rounded the next state to the nearest grid point, $(0.700,0.325)$. The patient was always at one of the $41^{2}=1681$ grid points, and a Q-table with one row per grid point held every state the patient could reach.
> * __This lab (not rounded):__ The patient stays at $(0.7035,0.3333)$, which lies between grid points. No table has a row for every state that the patient can reach. Instead, we estimate a row of parameters for each dose, and we compute the value of each dose as the inner product of its parameters and the features of the patient state.
>
> We now need to estimate the parameters of each dose.

Let's build the `patient::MyPatientModel` instance with [the `build(...)` function](src/Factory.jl):

In [ ]:
patient = build(MyPatientModel, (
    doses = [0.0, 1/3, 2/3, 1.0], # the dose u of actions 1 to 4
    g = 0.15,       # tumor growth rate per cycle
    k = 0.35,       # fraction of the tumor killed per cycle at full dose
    b = 0.50,       # toxicity added per cycle at full dose
    c = 0.35,       # fraction of the toxicity cleared per cycle
    Bcure = 0.10,   # cured when B ≤ Bcure
    Zlethal = 0.90, # toxic death when Z ≥ Zlethal
    wB = 1.0,       # penalty weight of the tumor burden B′ after a cycle
    wZ = 1.0,       # penalty weight of the toxicity Z′ after a cycle
    wu = 0.10,      # penalty weight of the dose u
    rcure = 60.0,   # reward for a cure
    rdead = -60.0,  # reward for a toxic death
));
γ = 0.95; # discount factor

[The `treat(...)` function](src/Compute.jl) plays the role of the __world__. Given the state $(B,Z)$ and an action $a$, it returns the next state $(B^{\prime},Z^{\prime})$, the reward $r$, and the done flag, which is `true` if the patient is cured or has died.

Let's treat a patient with a large tumor and no toxicity, $(B,Z)=(0.90,0)$, for one cycle with each of the four doses, and see what the world is telling us:

In [ ]:
let
    B = 0.90; # the starting tumor burden: a large tumor
    Z = 0.0;  # the starting toxicity: none
    number_of_actions = 4; # the number of doses available to the doctor
    grid_step = 0.025; # the spacing of the 41 grid levels of the dosing lab

    # one row per action: the dose, then what the world returns after one cycle -
    # the rounded columns hold the next state of the dosing lab: each value rounded to the nearest grid level -
    one_cycle = DataFrame(action = Int64[], dose = Float64[], B′ = Float64[], B′_rounded = Float64[],
        Z′ = Float64[], Z′_rounded = Float64[], r = Float64[], done = Bool[]);
    for a ∈ 1:number_of_actions
        (B′, Z′, r, done) = treat(patient, B, Z, a);
        B′_rounded = round(B′/grid_step)*grid_step; # the grid level nearest B′
        Z′_rounded = round(Z′/grid_step)*grid_step; # the grid level nearest Z′
        push!(one_cycle, (a, round(patient.doses[a], digits = 2), round(B′, digits = 4), round(B′_rounded, digits = 4),
            round(Z′, digits = 4), round(Z′_rounded, digits = 4), round(r, digits = 4), done));
    end

    # wow! That is indeed a really pretty table!
    pretty_table(one_cycle; table_format = TextTableFormat(borders = text_table_borders__compact), 
        fit_table_in_display_horizontally = false)
end

__What do we see?__ A larger dose shrinks the tumor more, but it also adds more toxicity. None of the four doses ends the treatment in one cycle.

The `B′` and `Z′` columns are the next state in this lab. The `B′_rounded` and `Z′_rounded` columns are the same values rounded to the nearest multiple of 0.025, as in the dosing lab. The row of action 3 is the cycle from the box above.

### Things to think about

__Question:__ Which action has the largest reward `r` in the table? What would happen to the tumor if we chose that action in every cycle?
___

## Task 2: Follow Grid Policies on the Continuous Patient
In this task, we test whether the method of the L12b dosing lab still works when the patient state is not rounded.

In the dosing lab, value iteration gave the best dose at each grid point. We call this table of doses a grid policy. [The `solvegrid(...)` function](src/Compute.jl) computes a grid policy: with 41 levels, it gives the optimal policy of the dosing lab, and with 201 levels, the grid is five times finer.

The patient is now between grid points. Each cycle, we look up the grid point nearest the patient state and give its dose. The patient then moves by the equations of Task 1, with no rounding.

Does a grid policy still cure the patient? Let's compute both grid policies and follow them from the dosing lab's starting state:

In [ ]:
grid_41 = solvegrid(patient; nlevels = 41, γ = γ);   # the dosing lab's grid: 41^2 = 1681 states
grid_201 = solvegrid(patient; nlevels = 201, γ = γ); # a finer grid: 201^2 = 40401 states
run_41 = simulate(patient, grid_41; B₀ = 0.90, Z₀ = 0.0, maxcycles = 100, γ = γ);
run_201 = simulate(patient, grid_201; B₀ = 0.90, Z₀ = 0.0, maxcycles = 100, γ = γ);

let
    # initialize --
    runs = [run_201, run_41]; # the treatment of each policy
    labels = ["201-level grid", "41-level grid"];
    colors = [:seagreen, :darkorange]; # one color per policy
    last_cycle = 40; # the 41-level policy runs to the cycle limit, so we draw only its first cycles
    p1 = plot(); # the tumor burden of each policy
    p2 = plot(); # the toxicity of each policy
    panels = []; # the tumor burden figure, the toxicity figure, then one small dose figure per policy

    # the tumor burden and the toxicity of each policy, cycle by cycle -
    for j ∈ 1:length(runs)
        n = min(runs[j].cycles, last_cycle); # the last cycle we draw for this policy
        cycles = range(0, stop = n, step = 1) |> collect; # the cycle numbers 0, 1, 2, ...
        plot!(p1, cycles, runs[j].B[1:(n + 1)], lw = 3, c = colors[j], label = labels[j]);
        plot!(p2, cycles, runs[j].Z[1:(n + 1)], lw = 3, c = colors[j], label = labels[j]);
    end

    # the level that ends the treatment in each figure -
    hline!(p1, [patient.Bcure], lw = 2, ls = :dash, c = :gray50, label = "cure level");
    hline!(p2, [patient.Zlethal], lw = 2, ls = :dash, c = :gray50, label = "lethal level");

    # labels and style: the gray panel is see-through, so the figure works in both themes -
    ylabel!(p1, "Tumor burden B");
    ylabel!(p2, "Toxicity Z");
    for p ∈ [p1, p2]
        xlabel!(p, "Treatment cycle");
        plot!(p, background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, fg_legend = :transparent,
            xlims = (-0.3, last_cycle + 0.3), ylims = (-0.03, 1.05));
    end
    plot!(p1, legend = :topright);    # the tumor burden falls, so the top right corner is empty
    plot!(p2, legend = :bottomright); # the toxicity rises, so the bottom right corner is empty
    push!(panels, p1);
    push!(panels, p2);

    # the dose of each policy in each cycle, in one small figure per policy -
    for j ∈ 1:length(runs)
        n = min(runs[j].cycles, last_cycle); # the last cycle we draw for this policy
        cycles = range(0, stop = n, step = 1) |> collect; # the cycle numbers 0, 1, 2, ...
        doses = [runs[j].dose[1:n]; runs[j].dose[n]]; # the dose of each cycle; the last value closes the step line

        # :steppost holds each dose flat until the next cycle -
        q = plot(cycles, doses, seriestype = :steppost, lw = 3, c = colors[j], label = "");

        # only the top figure gets a title, and only the bottom figure gets the cycle numbers and the axis label -
        if (j == 1)
            title!(q, "Dose u of each policy");
        end
        if (j == length(runs))
            xlabel!(q, "Treatment cycle");
        else
            plot!(q, xformatter = :none); # no cycle numbers under the upper figures
        end
        ylabel!(q, labels[j]);
        plot!(q, background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, titlefontsize = 11,
            yticks = ([0, 1/3, 2/3, 1], ["0", "1/3", "2/3", "1"]), xlims = (-0.3, last_cycle + 0.3), ylims = (-0.1, 1.1));
        push!(panels, q);
    end

    # the two state figures, then the dose figures stacked in a third column, all on the same cycle axis -
    layout = @layout [a{0.35w} b{0.35w} grid(2, 1)]; # each state figure gets 35% of the width, the dose column the rest
    plot(panels..., layout = layout, size = (1400, 460), bottom_margin = 5Plots.mm, left_margin = 6Plots.mm)
end

__What do we see?__ The 201-level policy cures the patient in 19 cycles, and its treatment ends. The 41-level policy never cures the patient, and its treatment runs to the limit of 100 cycles.

The orange line seems to touch the cure level, but it stays above the cure level. A cure needs $B\leq 0.10$, and the smallest tumor burden that the 41-level policy reaches is 0.1013.

In the dosing lab, the same policy cured the patient in 13 cycles. In the last cycle of that treatment, the tumor burden fell from 0.125 to 0.1122, and the world rounded 0.1122 to the grid level 0.10, a cure. In this lab, a tumor burden of 0.1122 is not rounded, and the patient is not cured.

The 41-level policy still chooses each dose from the rounded state. A tumor burden just above the cure level, such as 0.109, rounds to the grid level 0.100, which counts as a cure. A cured patient gets no dose, and the tumor grows back.

Let's check the dose that each grid policy gives at the state $(B,Z)=(0.109,0.816)$, which the 41-level policy reaches:

In [ ]:
let
    B = 0.109; # the tumor burden of a state the 41-level policy reaches
    Z = 0.816; # the toxicity of that state

    # the dose each grid policy gives at (B, Z) -
    doses_at_state = DataFrame(policy = ["41-level grid", "201-level grid"],
        dose = round.([patient.doses[gridaction(grid_41, B, Z)], patient.doses[gridaction(grid_201, B, Z)]], digits = 2));
    
    # wow! That is indeed a really pretty table!
    pretty_table(doses_at_state; table_format = TextTableFormat(borders = text_table_borders__compact), 
        fit_table_in_display_horizontally = false)
end

__What do we see?__ At this state, the 41-level policy gives no dose. The 201-level policy gives a dose, and the treatment goes on.

### The 41-level policy on the grid
Let's draw the 41-level policy on the $(B,Z)$ grid, with the path of the patient on top. [The `draw_grid_policy(...)` function](src/Visualize.jl) draws one square cell for each grid point. Every patient state inside a cell rounds to the grid point at its center, and the policy gives every state in the cell the same dose.

A darker blue cell is a larger dose. A green cell is a grid point that counts as a cure, and a red cell is a grid point that counts as a toxic death. The dashed lines mark the cure level and the lethal level of the patient model, and the black line is the first 40 cycles of the treatment:

In [ ]:
# draw the 41-level grid policy, with the first 40 cycles of its treatment on top -
draw_grid_policy(patient, grid_41, run_41; label = "41-level policy", last_cycle = 40)

__What do we see?__ Each line segment is one treatment cycle. A cycle with a dose shrinks the tumor and raises the toxicity, which moves the patient up and to the left. A cycle with no dose lets the toxicity clear and the tumor grow, which moves the patient down and to the right.

The policy stops the doses too early. A cure needs a tumor burden of 0.10 or less. The green cells go up to 0.1125, because every tumor burden up to 0.1125 rounds to the grid level 0.10.

When the tumor burden is between 0.10 and 0.1125, the patient gets no dose and is not cured. The tumor grows, the policy gives a dose again, and the tumor shrinks back into the green cells. This repeats until the cycle limit.

__Why does the 201-level policy cure the patient?__ The 201-level grid has the same error in a narrower range. Its grid step is 0.005, and only a tumor burden between 0.10 and 0.1025 rounds to the grid level 0.10.

The two policies differ in the dose that follows a cycle with no dose. On the 41-level grid, the 2/3 dose takes the grid level 0.125 to 0.1122, which rounds to a cure. The 41-level policy gives the 2/3 dose, which does not cure the patient.

On the 201-level grid, the 2/3 dose takes the grid level 0.115 to 0.1034, which does not round to a cure. The 201-level policy gives the full dose, which takes 0.115 to 0.090, a cure.

### Things to think about
Let's compute a grid policy for each number of levels in `nlevels_values` and follow each one from the dosing lab's starting state:

In [ ]:
let
    nlevels_values = [41, 49, 81, 201]; # the number of grid levels on each axis

    # for each grid: value iteration, then follow its policy on the patient without a grid -
    grids = DataFrame(levels = Int64[], outcome = Symbol[], cycles = Int64[]);
    for n ∈ nlevels_values
        grid_n = solvegrid(patient; nlevels = n, γ = γ);
        run_n = simulate(patient, grid_n; B₀ = 0.90, Z₀ = 0.0, maxcycles = 100, γ = γ);
        push!(grids, (n, run_n.outcome, run_n.cycles));
    end
    pretty_table(grids; backend = :text, fit_table_in_display_horizontally = false)
end

__Question:__ Which grids cure the patient? Add a few values to `nlevels_values` and rerun the cell: does a finer grid always give a better treatment?
___

## Task 3: Train a Linear DQN
In this task, we build a linear Q-network, train it with DQN, and compare its greedy policy with the grid policies.

The Q-value of action $a$ in the state $s=(B,Z)$ is the discounted return the agent expects after taking action $a$. The network computes the Q-value as a weighted sum of features:

$$
\begin{equation*}
Q_{\mathbf{W}}(s,a) = \mathbf{w}_{a}^{\top}\phi(s),
\end{equation*}
$$

where $\mathbf{w}_{a}^{\top}$ is row $a$ of the weight matrix $\mathbf{W}$, and $\phi(s)$ is the feature vector of the state. Our features are radial basis functions. We place the centers $\mathbf{c}_{j}$ of $m\times m$ basis functions on an even grid over the square $0\leq B\leq 1$, $0\leq Z\leq 1$. Basis function $j$ is given by:

$$
\begin{equation*}
\phi_{j}(s)=\exp\left(-\frac{\lVert s-\mathbf{c}_{j}\rVert_{2}^{2}}{2\sigma^{2}}\right),
\end{equation*}
$$

where $\lVert s-\mathbf{c}_{j}\rVert_{2}$ is the Euclidean distance from the state to center $j$, and $\sigma$ is the width of every basis function. A last feature is the constant 1, which lets each action's Q-value shift up or down by the same amount in every state. Each action then has $m^{2}+1$ weights.

The grid of centers only places the basis functions. The network can evaluate any state, not only the centers. Let's build the network with [the `build(...)` function](src/Factory.jl), with all its weights at zero.

Let's also plot the six basis functions with centers at zero toxicity, along the tumor burden axis. The three panels use a narrow width, our width, and a wide width.

Each curve is one basis function, and each × marks its center. The dashed line is the state $(B,Z)=(0.90,0)$. Each red dot is the value of one basis function at that state, which is one feature of the state:

In [ ]:
network = build(MyLinearQNetworkModel, (nactions = 4, m = 6, σ = 0.15)); # 4 actions, 6 × 6 basis functions of width 0.15
φ = features(network, 0.90, 0.0); # the features of the state B = 0.90, Z = 0

# the basis functions along the tumor burden axis at Z = 0, for a narrow width, our width, and a wide width -
let
    B_values = range(0.0, 1.0, length = 201) |> collect; # tumor burdens from 0 to 1
    row = findall(network.centers[:, 2] .== 0.0); # the six basis functions with centers on the row Z = 0
    centers_B = network.centers[row, 1]; # the tumor burden of each of these centers
    σ_values = [0.05, network.σ, 0.5]; # the width of each panel
    σ_names = ["narrow", "our width", "wide"];
    panels = [];
    for k ∈ 1:length(σ_values)
        σ = σ_values[k];
        network_σ = build(MyLinearQNetworkModel, (nactions = 4, m = 6, σ = σ));
        φ_values = [features(network_σ, B, 0.0) for B ∈ B_values]; # the features at each tumor burden
        p = plot(xlabel = "tumor burden B (Z = 0)", ylabel = "basis function value", ylims = (0, 1.55), yticks = 0:0.25:1,
            title = "σ = $(σ) ($(σ_names[k]))", titlefontsize = 11, legend = false);

        # one curve for each basis function; only the first curve gets a legend entry -
        for j ∈ row
            curve_label = "";
            if (j == row[1])
                curve_label = "basis function";
            end
            plot!(p, B_values, [φ_B[j] for φ_B ∈ φ_values], lw = 2, c = :dodgerblue3, label = curve_label);
        end

        # an × at the center of each basis function, where the basis function equals 1 -
        scatter!(p, centers_B, ones(length(row)), marker = :x, ms = 5, msw = 2, c = :darkorange, label = "center");

        # the state B = 0.90, and a red dot at the value of each basis function there -
        vline!(p, [0.90], c = :gray, ls = :dash, label = "state B = 0.90");
        scatter!(p, fill(0.90, length(row)), features(network_σ, 0.90, 0.0)[row], c = :red, ms = 4, label = "feature of the state");
        push!(panels, p);
    end

    # the legend goes in the first panel, in the empty space above the curves -
    plot!(panels[1], legend = :topleft, legendfontsize = 8, fg_legend = :transparent);
    plot(panels..., layout = (1, 3), size = (960, 340), margin = 5Plots.mm)
end

__What do we see?__ Each basis function equals 1 at its center and decays toward 0 as the distance from the center grows. With our width, $\sigma=0.15$ in the middle panel, only the basis functions centered near the state are large at the state, and the rest are close to zero. A gradient step at this state then changes mostly the weights of the basis functions centered nearby.

The width $\sigma$ sets how fast each basis function decays. With the narrow width, $\sigma=0.05$ in the left panel, every basis function is small at a state between two centers. With the wide width, $\sigma=0.5$ in the right panel, every basis function is large over most of the axis.

The width matters because the Q-value is the inner product $Q_{\mathbf{W}}(s,a)=\mathbf{w}_{a}^{\top}\phi(s)$. The weight vector $\mathbf{w}_{a}$ is the same at every state. The Q-value changes with the state only through the feature vector $\phi(s)$.

* __Narrow width, $\sigma=0.05$ (left panel):__ At a state between two centers, every basis function is small. Then $\mathbf{w}_{a}^{\top}\phi(s)$ is nearly the same at all of these states, and the network cannot tell them apart.
* __Our width, $\sigma=0.15$ (middle panel):__ Nearby states have similar feature vectors, and distant states have different feature vectors. Then $\mathbf{w}_{a}^{\top}\phi(s)$ gives nearby states similar Q-values, and it can give distant states different Q-values.
* __Wide width, $\sigma=0.5$ (right panel):__ The feature vector $\phi(s)$ changes slowly with the state. Then $\mathbf{w}_{a}^{\top}\phi(s)$ changes slowly with the state too, and it cannot match a Q-value that changes sharply near the cure level or the lethal level.

### Train the network
[The `mydqn(...)` function](src/Compute.jl) carries out the DQN algorithm of the lecture. Each episode is one simulated treatment that starts the patient at a random state. Random starts spread the agent's experience over many states.

The keyword arguments of `mydqn(...)` set the training:
* The `episodes` argument is the number of simulated treatments, and `maxcycles` is the maximum number of cycles in each treatment.
* The `ϵ_start` and `ϵ_floor` arguments set the exploration probability $\epsilon$, the chance that a cycle gives a random dose. The probability falls in a straight line from `ϵ_start` and then stays at `ϵ_floor`.
* The `capacity` argument is the size of the replay buffer. Training starts once the buffer holds `warmup` transitions.
* The `batchsize` argument is the mini-batch size $N_{\text{batch}}$, and `α` is the learning rate of each gradient step.
* The `C` argument is the number of cycles between copies of the Q-network to the target network.

Each gradient step uses a mini-batch of $N_{\text{batch}}$ transitions. Transition $i$ holds the state $s_{i}$, the action $a_{i}$, the reward $r_{i}$, the next state $s_{i}^{\prime}$, and the done flag $d_{i}$, which is 1 for a cure or a toxic death and 0 otherwise. The step is gradient descent on the mini-batch loss of the lecture:

$$
\begin{align*}
\mathcal{L}(\mathbf{W}) & = \frac{1}{2N_{\text{batch}}}\sum_{i=1}^{N_{\text{batch}}}\left(y_{i}-\mathbf{w}_{a_{i}}^{\top}\phi(s_{i})\right)^{2}, \\
y_{i} & = r_{i}+\gamma\,(1-d_{i})\max_{a^{\prime}}Q_{\mathbf{W}^{-}}(s_{i}^{\prime},a^{\prime}),
\end{align*}
$$

where $y_{i}$ is the target of transition $i$, the maximum is over the four actions $a^{\prime}$, and $\mathbf{W}^{-}$ holds the weights of the target network. The factor $(1-d_{i})$ drops the future value when the treatment has ended. The gradient step updates row $a$ of the weight matrix by:

$$
\begin{equation*}
\mathbf{w}_{a}\gets\mathbf{w}_{a}+\frac{\alpha}{N_{\text{batch}}}\sum_{i\,:\,a_{i}=a}\left(y_{i}-\mathbf{w}_{a}^{\top}\phi(s_{i})\right)\phi(s_{i}),
\end{equation*}
$$

where the sum runs over the transitions of the mini-batch that took action $a$.

> __Why is the target $y_{i}$ not just the reward $r_{i}$?__ The reward $r_{i}$ scores one cycle. The Q-value scores the treatment from this cycle on: the reward of this cycle plus the discounted rewards of the later cycles, when every later dose is the best dose. In Task 1, no dose had the largest one-cycle reward, but no dose in every cycle never cures the patient.
>
> We observe the reward of this cycle. We do not know the rewards of the later cycles under the best doses, and we estimate them with the target network at the next state:
>
> $$y_{i} = \underbrace{r_{i}}_{\text{this cycle: observed}}+\underbrace{\gamma\,(1-d_{i})\max_{a^{\prime}}Q_{\mathbf{W}^{-}}(s_{i}^{\prime},a^{\prime})}_{\text{later cycles: estimated}}.$$
>
> When the treatment ends, $d_{i}=1$ and there are no later cycles. The target is then the reward alone, $y_{i}=r_{i}$. The optimal Q-values $Q^{*}$ satisfy the same equation with $Q^{*}$ on both sides, which is the Bellman equation:
>
> $$Q^{*}(s_{i},a_{i}) = r_{i}+\gamma\,(1-d_{i})\max_{a^{\prime}}Q^{*}(s_{i}^{\prime},a^{\prime}).$$
>
> If the predictions matched the targets at every state and action, with $\mathbf{W}^{-}=\mathbf{W}$, the network would satisfy the Bellman equation and equal $Q^{*}$.
>
> __Why do we need the target network $\mathbf{W}^{-}$?__ The target $y_{i}$ uses Q-values of the next state $s_{i}^{\prime}$. If we computed them with $\mathbf{W}$, the weights we are training, each gradient step would change the targets as well as the predictions. The predictions and the targets could then grow together without bound.
>
> The target network uses $\mathbf{W}^{-}$, a copy of $\mathbf{W}$ that we replace only every $C$ cycles. A gradient step changes $\mathbf{W}$ but not $\mathbf{W}^{-}$. The predictions change, and the targets do not.

The `mydqn(...)` function records the loss of each gradient step, computed before the step. Let's train the network and plot the loss against the gradient step, on a log scale.

The gray line is the loss of each gradient step. The loss of one gradient step is noisy. To show the trend, we group the gradient steps into blocks of 500 consecutive steps, and the blue line is the mean loss of each block:

In [ ]:

# run the training of the linear Q-network on the patient, and return the trained network and the loss of each gradient step -
result = mydqn(network, patient;
    episodes = 3000,    # number of simulated treatments (episodes)
    maxcycles = 80,     # most treatment cycles in one episode
    γ = γ,              # discount factor
    α = 0.05,           # learning rate of each gradient step
    capacity = 20_000,  # number of transitions the replay buffer can hold
    batchsize = 32,     # number of transitions in a mini-batch
    warmup = 1000,      # number of transitions in the buffer before the gradient steps start
    C = 500,            # number of cycles between copies of the Q-network to the target network
    ϵ_start = 1.0,      # exploration probability ϵ at the start; it falls in a straight line to ϵ_floor
    ϵ_floor = 0.05,     # smallest exploration probability ϵ
    seed = 1234         # seed of the random number generator, so a rerun gives the same result
);
trained = result.network; # the trained network

# the loss of each gradient step, and its average over blocks of gradient steps -
let
    loss = result.loss; # the mini-batch loss of each gradient step
    steps = range(1, stop = length(loss), step = 1) |> collect; # the gradient steps 1, 2, 3, ...
    block = 500; # the number of gradient steps in one block
    number_of_blocks = length(loss) ÷ block;
    block_steps = [k*block for k ∈ 1:number_of_blocks]; # the last gradient step of each block
    block_loss = [mean(loss[((k - 1)*block + 1):(k*block)]) for k ∈ 1:number_of_blocks]; # the average loss of each block

    # the loss of each step in gray, and the block average on top -
    plot(steps, loss, lw = 1, c = :gray50, alpha = 0.35, label = "loss of each gradient step");
    plot!(block_steps, block_loss, lw = 3, c = :dodgerblue3, label = "mean of each block of $(block) steps");

    # labels and style: the gray panel is see-through, so the figure works in both themes -
    xlabel!("Gradient step");
    ylabel!("Mini-batch loss");
    plot!(yscale = :log10, ylims = (0.2, 400), yticks = ([1, 10, 100], ["1", "10", "100"]), xformatter = :plain,
        background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, fg_legend = :transparent,
        legend = :topright, size = (760, 460));
end

__What do we see?__ The loss of a single gradient step varies widely from step to step, because each mini-batch holds different random transitions. The mean loss of each block falls over the training run. It is still falling at the end of the run.

Would a longer run lower the loss further? Let's train with more episodes and plot the mean loss of each block of 500 gradient steps, for each run:

In [ ]:
let
    episodes_values = [3000, 6000, 12000]; # the number of episodes of each training run
    colors = [:dodgerblue3, :seagreen, :darkorange]; # one color per training run
    block = 500; # the number of gradient steps in one block

    # for each run: train a network, then plot the average loss of each block of gradient steps -
    plot();
    for j ∈ 1:length(episodes_values)
        result_j = mydqn(network, patient;
            episodes = episodes_values[j], # number of simulated treatments (episodes) of this training run
            maxcycles = 80,     # most treatment cycles in one episode
            γ = γ,              # discount factor
            α = 0.05,           # learning rate of each gradient step
            capacity = 20_000,  # number of transitions the replay buffer can hold
            batchsize = 32,     # number of transitions in a mini-batch
            warmup = 1000,      # number of transitions in the buffer before the gradient steps start
            C = 500,            # number of cycles between copies of the Q-network to the target network
            ϵ_start = 1.0,      # exploration probability ϵ at the start; it falls in a straight line to ϵ_floor
            ϵ_floor = 0.05,     # smallest exploration probability ϵ
            seed = 1234         # seed of the random number generator, so a rerun gives the same result
        );
        loss = result_j.loss; # the mini-batch loss of each gradient step
        number_of_blocks = length(loss) ÷ block;
        block_steps = [k*block for k ∈ 1:number_of_blocks]; # the last gradient step of each block
        block_loss = [mean(loss[((k - 1)*block + 1):(k*block)]) for k ∈ 1:number_of_blocks]; # the average loss of each block
        plot!(block_steps, block_loss, lw = 3, c = colors[j], label = "$(episodes_values[j]) episodes");
    end

    # labels and style: the gray panel is see-through, so the figure works in both themes -
    xlabel!("Gradient step");
    ylabel!("Mean mini-batch loss of each block");
    plot!(yscale = :log10, ylims = (1.5, 100), yticks = ([2, 5, 10, 20, 50], ["2", "5", "10", "20", "50"]), xformatter = :plain,
        background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, fg_legend = :transparent,
        legend = :topright, size = (760, 460));
end

__What do we see?__ A longer run lowers the loss a little more, and then the loss levels off. The loss does not fall to zero. The targets change each time the Q-network is copied to the target network, and a weighted sum of basis functions cannot match every target exactly.

A longer run also explores for more episodes, because the exploration probability falls over a fixed fraction of the episodes. The loss of a longer run then falls more slowly.

### Follow the learned policy
We now treat the patient with the trained network. In each cycle, we give the dose with the largest Q-value at the patient's state, and never a random dose. We call this rule the learned policy.

[The `simulate(...)` function](src/Compute.jl) gives the doses of the learned policy, cycle by cycle, until the patient is cured, the patient dies of toxicity, or the treatment reaches 100 cycles.

Let's follow the learned policy from the dosing lab's starting state, $(B,Z)=(0.90,0)$, and plot the tumor burden, the toxicity, and the dose in each cycle:

In [ ]:
run_dqn = simulate(patient, trained; B₀ = 0.90, Z₀ = 0.0, maxcycles = 100, γ = γ);

let
    cycles = range(0, stop = run_dqn.cycles, step = 1) |> collect; # the cycle numbers 0, 1, 2, ...
    doses = [run_dqn.dose; run_dqn.dose[end]]; # the dose of each cycle; the last value closes the step line

    # the tumor burden and the toxicity after each cycle -
    plot(cycles, run_dqn.B, lw = 3, c = :dodgerblue3, marker = :circle, ms = 4, msc = :dodgerblue3, label = "tumor burden B");
    plot!(cycles, run_dqn.Z, lw = 3, c = :red, marker = :circle, ms = 4, msc = :red, label = "toxicity Z");

    # the dose in each cycle, in the color of the legend text: black on the light theme, white on the dark theme -
    dose_color = current()[1][:legend_font_color];
    plot!(cycles, doses, seriestype = :steppost, lw = 2, ls = :dash, c = dose_color, label = "dose u");

    # the levels that end the treatment -
    hline!([patient.Bcure], lw = 2, ls = :dash, c = :dodgerblue3, label = "cure level");
    hline!([patient.Zlethal], lw = 2, ls = :dash, c = :red, label = "lethal level");

    # labels and style: the gray panel is see-through, so the figure works in both themes -
    xlabel!("Treatment cycle");
    ylabel!("Tumor burden, toxicity, and dose");
    plot!(background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, fg_legend = :transparent,
        legend = :outertop, legend_column = 3, xlims = (-0.3, run_dqn.cycles + 0.3), ylims = (-0.03, 1.05), size = (760, 460));
end

__What do we see?__ The learned policy cures the patient. It gives mostly moderate doses and only a few full doses, never two in a row. The toxicity rises after each large dose and falls after each small one, and it stays below the lethal level while the tumor shrinks.

### Compare with the grid policies
Let's plot the treatment of the linear DQN with the treatments of the two grid policies, from the same starting state:

In [ ]:
let
    # initialize --
    runs = [run_dqn, run_201, run_41]; # the treatment of each policy
    labels = ["linear DQN", "201-level grid", "41-level grid"];
    colors = [:dodgerblue3, :seagreen, :darkorange]; # one color per policy
    last_cycle = 40; # the 41-level policy runs to the cycle limit, so we draw only its first cycles
    p1 = plot(); # the tumor burden of each policy
    p2 = plot(); # the toxicity of each policy
    panels = []; # the tumor burden figure, the toxicity figure, then one small dose figure per policy

    # the tumor burden and the toxicity of each policy, cycle by cycle -
    for j ∈ 1:length(runs)
        n = min(runs[j].cycles, last_cycle); # the last cycle we draw for this policy
        cycles = range(0, stop = n, step = 1) |> collect; # the cycle numbers 0, 1, 2, ...
        plot!(p1, cycles, runs[j].B[1:(n + 1)], lw = 3, c = colors[j], label = labels[j]);
        plot!(p2, cycles, runs[j].Z[1:(n + 1)], lw = 3, c = colors[j], label = labels[j]);
    end

    # the level that ends the treatment in each figure -
    hline!(p1, [patient.Bcure], lw = 2, ls = :dash, c = :gray50, label = "cure level");
    hline!(p2, [patient.Zlethal], lw = 2, ls = :dash, c = :gray50, label = "lethal level");

    # labels and style: the gray panel is see-through, so the figure works in both themes -
    ylabel!(p1, "Tumor burden B");
    ylabel!(p2, "Toxicity Z");
    for p ∈ [p1, p2]
        xlabel!(p, "Treatment cycle");
        plot!(p, background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, fg_legend = :transparent,
            xlims = (-0.3, last_cycle + 0.3), ylims = (-0.03, 1.05));
    end
    plot!(p1, legend = :topright);    # the tumor burden falls, so the top right corner is empty
    plot!(p2, legend = :bottomright); # the toxicity rises, so the bottom right corner is empty
    push!(panels, p1);
    push!(panels, p2);

    # the dose of each policy in each cycle, in one small figure per policy -
    for j ∈ 1:length(runs)
        n = min(runs[j].cycles, last_cycle); # the last cycle we draw for this policy
        cycles = range(0, stop = n, step = 1) |> collect; # the cycle numbers 0, 1, 2, ...
        doses = [runs[j].dose[1:n]; runs[j].dose[n]]; # the dose of each cycle; the last value closes the step line

        # :steppost holds each dose flat until the next cycle -
        q = plot(cycles, doses, seriestype = :steppost, lw = 3, c = colors[j], label = "");

        # only the top figure gets a title, and only the bottom figure gets the cycle numbers and the axis label -
        if (j == 1)
            title!(q, "Dose u of each policy");
        end
        if (j == length(runs))
            xlabel!(q, "Treatment cycle");
        else
            plot!(q, xformatter = :none); # no cycle numbers under the upper figures
        end
        ylabel!(q, labels[j]);
        plot!(q, background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, titlefontsize = 11,
            yticks = ([0, 1/3, 2/3, 1], ["0", "1/3", "2/3", "1"]), xlims = (-0.3, last_cycle + 0.3), ylims = (-0.1, 1.1));
        push!(panels, q);
    end

    # the two state figures, then the dose figures stacked in a third column, all on the same cycle axis -
    layout = @layout [a{0.35w} b{0.35w} grid(3, 1)]; # each state figure gets 35% of the width, the dose column the rest
    plot(panels..., layout = layout, size = (1400, 460), bottom_margin = 5Plots.mm, left_margin = 6Plots.mm)
end

__What do we see?__ The linear DQN and the 201-level policy both cure the patient. The 201-level policy needs fewer cycles. Its toxicity also comes closer to the lethal level.

### Things to think about
The width $\sigma$ sets how fast each basis function decays with the distance from its center. Let's train one network for each width in `σ_values` and follow each learned policy from the same start:

In [ ]:
let
    σ_values = [0.05, 0.15, 0.5]; # the widths to compare: narrow, the width we trained with, and wide

    # for each width: build and train a network, then follow its greedy policy -
    widths = DataFrame(σ = Float64[], outcome = Symbol[], cycles = Int64[]);
    for σ ∈ σ_values
        network_σ = build(MyLinearQNetworkModel, (nactions = 4, m = 6, σ = σ));
        result_σ = mydqn(network_σ, patient;
            episodes = 3000,    # number of simulated treatments (episodes)
            maxcycles = 80,     # most treatment cycles in one episode
            γ = γ,              # discount factor
            α = 0.05,           # learning rate of each gradient step
            capacity = 20_000,  # number of transitions the replay buffer can hold
            batchsize = 32,     # number of transitions in a mini-batch
            warmup = 1000,      # number of transitions in the buffer before the gradient steps start
            C = 500,            # number of cycles between copies of the Q-network to the target network
            ϵ_start = 1.0,      # exploration probability ϵ at the start; it falls in a straight line to ϵ_floor
            ϵ_floor = 0.05,     # smallest exploration probability ϵ
            seed = 1234         # seed of the random number generator, so a rerun gives the same result
        );
        run_σ = simulate(patient, result_σ.network; B₀ = 0.90, Z₀ = 0.0, maxcycles = 100, γ = γ);
        push!(widths, (σ, run_σ.outcome, run_σ.cycles));
    end
    pretty_table(widths; backend = :text, fit_table_in_display_horizontally = false)
end

__Question:__ Which widths cure the patient? Use the basis function figure to explain why a width that is too narrow or too wide fails.
___

## Summary
In this lab, we learned a dosing policy for the patient of the dosing lab with a linear DQN, without rounding the patient state to a grid.

> __Key Takeaways:__
>
> * __Rounding can give the wrong dose:__ We computed grid policies by value iteration and followed them on a patient whose state is not rounded. The dosing lab's grid rounded a tumor burden just above the cure level to a cure and stopped the doses, and one finer grid gave a lethal dose.
>
> * __A linear Q-network replaces the table:__ We built a Q-network that computes the value of each dose as a weighted sum of radial basis functions of the tumor burden and the toxicity. It gives a dose for every patient state, with no grid and no rounding, and it needs far fewer numbers than a Q-table on a fine grid.
>
> * __Training the network with DQN:__ We trained the network on simulated treatments, with a replay buffer and a target network. The greedy policy cured the patient from the dosing lab's starting state and kept the toxicity below the lethal level.

DQN helps us with continuous states, but the actions are still discrete. This is often fine, because many decisions are categorical. For continuous states and actions, we need other methods, such as the policy gradient and actor-critic methods of [Sutton and Barto, 2018, Chapter 13](http://incompleteideas.net/book/the-book-2nd.html).

___